In [1]:
from data import om_oilseed

In [2]:
print('历史数据入库')
om_oilseed.save_archive()

历史数据入库
.. 历史数据量：224409; 
.. 库内最新日期：2025-10-11; 
.. 请求日期区间：2025-10-08~2025-10-11; 
.. 实际写入区间：2025-10-08~2025-10-11; 
.. 写入成功


In [3]:
print('读取历史数据')
archive_df = om_oilseed.read_archive()

读取历史数据
.. 历史数据量：224409; 


In [4]:
print('获取预测数据')
forecast_df = om_oilseed.get_forecast(archive_df['date'].max())

获取预测数据
.. 请求日期区间：2025-10-12~2025-10-26; 
.. 预测数据量：855; 


In [5]:
print('数据加工（不偏移）')
process_df = om_oilseed.data_process(archive_df, forecast_df)

数据加工（不偏移）
.. 处理后数据量：55286; 


In [6]:
print('数据加工（偏移212天，即8月1日为每年第一天）')
offset = 212
process_offset_df = om_oilseed.data_process_offset(offset, archive_df, forecast_df)
process_offset_df = process_offset_df[ process_offset_df['date'] >= '2015-08-01']

from src import date_process
month_ticks, month_labels = date_process.dayofyear_offset_label(offset)

数据加工（偏移212天，即8月1日为每年第一天）
.. 处理后数据量：55286; 


In [7]:
print('绘制图像')
cities_df = om_oilseed.read_cities()
charts_df = om_oilseed.read_charts()
forecast_after = archive_df['date'].max()

# 制图
import src.plt_charts as charts
import matplotlib.pyplot as plt
for i in cities_df.index:
    # 城市参数
    country = cities_df.loc[i]['country']
    city = cities_df.loc[i]['city']
    tag = cities_df.loc[i]['tag']
    code = cities_df.loc[i]['code']
    df = process_df[ process_df['city_code'] == code ].copy()
    if country != 'Canada':
        df = process_offset_df[ process_offset_df['city_code'] == code ].copy()

    for j in charts_df.index:
        params = {
            'min_history_year': charts_df.loc[j]['min_history_year'],
            'forecast_after':forecast_after,
            'ylabel': charts_df.loc[j]['y_label'],
            'title': charts_df.loc[j]['title'] + city + ', ' + country
        }
        if country != 'Canada':
            params['month_ticks'] = month_ticks
            params['month_labels'] = month_labels
        chart = charts.day_annul_plot(df, charts_df.loc[j]['variable'], **params)

        path = f'./charts/oilseed/{i:02d}_{code}_{j:02d}_{charts_df.loc[j]['variable']}.jpg'
        chart.savefig(path, dpi=300)
        plt.close()

绘制图像


In [10]:
print('合成大图')
file_lists = []
for i in cities_df.index:
    code = cities_df.loc[i]['code']
    for j in charts_df.index:
        path = f'./charts/oilseed/{i:02d}_{code}_{j:02d}_{charts_df.loc[j]['variable']}.jpg'
        file_lists.append(path)
charts.merge2grid(file_lists, cities_df.shape[0], charts_df.shape[0], './charts/oilseed/merge_oilseed.jpg')

合成大图
